In [ ]:

!pip install odfpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 717.0/717.0 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for odfpy: filename=odfpy-1.4.1-py2.py3-none-any.whl size=160673 sha256=e86d79f8d4e11f3708f9e5c210e2d47f5cde0f0a24d38dd72bccb90c00546e70
  Stored in directory: /root/.cache/pip/wheels/36/5d/63/8243a7ee78fff0f944d638fd0e66d7278888f5e2285d7346b6
Successfully built odfpy


In [ ]:
import pandas as pd

# 1. Define the exact column names for this dataset
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

df = pd.read_csv('/content/adult.data', names=columns, skipinitialspace=True)

df.to_excel('adult.ods', engine='odf', index=False)

print("Conversion complete! You now have a clean 'adult.ods' file with proper headers.")

Conversion complete! You now have a clean 'adult.ods' file with proper headers.


In [ ]:
from odf.opendocument import load
from odf.table import Table, TableCell, TableRow
from odf.text import P
from odf import teletype

filename = 'adult.ods'
doc = load(filename)

# extracting 1st sheet
sheets = doc.getElementsByType(Table)
sheet = sheets[0]
sheet_name = sheet.getAttribute('name')
rows = doc.getElementsByType(TableRow)

header_rows = rows[0]
headers = [
    teletype.extractText(cell.getElementsByType(P)[0]).strip()
    if cell.getElementsByType(P) else ''
    for cell in header_rows.getElementsByType(TableCell)
]

print(f'Sheet Name: {sheet_name}')
print(f'Headers: {headers}')


Sheet Name: Sheet1
Headers: ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']


**Displaying samples Rows**

In [ ]:
print('Sample Rows: ')
for i, row in enumerate(rows[1:6], 1):
# for i, row in enumerate(rows[1:6], 1):
  cells = row.getElementsByType(TableCell)

  values = [
      teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
      for cell in cells
  ]

  print(f'Rows {i}: {values}')

Sample Rows: 
Rows 1: ['39', 'State-gov', '77516', 'Bachelors', '13', 'Never-married', 'Adm-clerical', 'Not-in-family', 'White', 'Male', '2174', '', '40', 'United-States', '<=50K']
Rows 2: ['50', 'Self-emp-not-inc', '83311', 'Bachelors', '13', 'Married-civ-spouse', 'Exec-managerial', 'Husband', 'White', 'Male', '', '', '13', 'United-States', '<=50K']
Rows 3: ['38', 'Private', '215646', 'HS-grad', '9', 'Divorced', 'Handlers-cleaners', 'Not-in-family', 'White', 'Male', '', '', '40', 'United-States', '<=50K']
Rows 4: ['53', 'Private', '234721', '11th', '7', 'Married-civ-spouse', 'Handlers-cleaners', 'Husband', 'Black', 'Male', '', '', '40', 'United-States', '<=50K']
Rows 5: ['28', 'Private', '338409', 'Bachelors', '13', 'Married-civ-spouse', 'Prof-specialty', 'Wife', 'Black', 'Female', '', '', '40', 'Cuba', '<=50K']


# **Data Cleaning**

In [ ]:
clean_headers = [h.strip().lower().replace(' ', '_') for h in headers]

clean_data = []
for row in rows[1:]:
  cells = row.getElementsByType(TableCell)
  if not cells or all(len(cell.getElementsByType(P)) == 0 for cell in cells):
    continue

  values = [
      teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
        for cell in cells
    ]
    # pad values if row is short
    # pad value in this context mean making the length of rows equal
  if len(values)< len(clean_headers):
      values += [''] * (len(clean_headers) - len(values))

    # Converts empry string to none
  values = [v if v != '' else None for v in values]
  record = dict(zip(clean_headers, values))
  clean_data.append(record)

print(f"Loaded and cleaned {len(clean_data)} records.")
print("Sample cleaned record:")
print(clean_data[0])

Loaded and cleaned 32561 records.
Sample cleaned record:
{'age': '39', 'workclass': 'State-gov', 'fnlwgt': '77516', 'education': 'Bachelors', 'education_num': '13', 'marital_status': 'Never-married', 'occupation': 'Adm-clerical', 'relationship': 'Not-in-family', 'race': 'White', 'sex': 'Male', 'capital_gain': '2174', 'capital_loss': None, 'hours_per_week': '40', 'native_country': 'United-States', 'income': '<=50K'}


# **Saving the clean dataset to ODS**

In [ ]:
from odf.opendocument import OpenDocumentSpreadsheet
from odf.table import Table, TableCell, TableRow
from odf.text import P

cleaned_file = 'cleaned_adult.ods'
doc_clean = OpenDocumentSpreadsheet()
table_clean = Table(name= 'CleanedAdult')

# header row
header_row = TableRow()
for h in clean_headers:
  cell = TableCell()
  cell.addElement(P(text=h))
  header_row.addElement(cell)
table_clean.addElement(header_row)

# Data Rows
for rec in clean_data:
  row = TableRow()
  for h in clean_headers:
    val = rec.get(h, '')
    cell = TableCell()
    cell.addElement(P(text=str(val) if val is not None else ''))
    row.addElement(cell)
  table_clean.addElement(row)

# save the cleaned file
doc_clean.spreadsheet.addElement(table_clean)
doc_clean.save(cleaned_file)
print(f'Cleaned file saved as {cleaned_file}')

Cleaned file saved as cleaned_adult.ods


# ***Question 1***

## **What is the distribution of education level in the dataset**

In [ ]:
from collections import Counter

education_count = Counter(rec['education'] for rec in clean_data if rec.get('education'))
print(f'Education Level Distribution:')
for edu, count in education_count.most_common():
  print(f'{edu}: {count}')

Education Level Distribution:
HS-grad: 10501
Some-college: 7291
Bachelors: 5355
Masters: 1723
Assoc-voc: 1382
11th: 1175
Assoc-acdm: 1067
10th: 933
7th-8th: 646
Prof-school: 576
9th: 514
12th: 433
Doctorate: 413
5th-6th: 333
1st-4th: 168
Preschool: 51


# **Question 2**

**What is the average age of individuals for each** **workclass**

In [ ]:
from collections import defaultdict

workclass_ages = defaultdict(list)
for rec in clean_data:
  workclass = rec.get('workclass')
  try:
    age = float(rec.get('age', 0) or 0)
    if workclass and age > 0:
      workclass_ages[workclass].append(age)
  except Exception:
      continue

print(f'Average age by workclass: ')
for workclass, ages in workclass_ages.items():
    avg_age = sum(ages) / len(ages) if ages else 0
    print(f'{workclass}: {avg_age:.2f}')


Average age by workclass: 
State-gov: 39.44
Self-emp-not-inc: 44.97
Private: 36.80
Federal-gov: 42.59
Local-gov: 41.75
?: 40.96
Self-emp-inc: 46.02
Without-pay: 47.79
Never-worked: 20.57


# **Question 3**

**What percentage of individuals earn more than 50k per year**

In [ ]:
total = 0
above_50k = 0
for rec in clean_data:
  income = rec.get('income') or rec.get('class')
  if income:
    total += 1
    if '>50K' in income.replace(' ', '') or '>50K' in income.replace(' ', ''):
      above_50k += 1

if total > 0:
  percent = (above_50k /total) *100
  print(f"Percentage earning >50K: {percent:.2f}%")
else:
  print('No income data found')

Percentage earning >50K: 24.08%


# **Task 1: Find average hours worked per week for each education category**

In [ ]:
from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf.text import P
from odf import teletype

filename = 'cleaned_adult.ods'
doc = load(filename)

# Extract the first sheet
sheets = doc.getElementsByType(Table)
sheet = sheets[0]
rows = sheet.getElementsByType(TableRow)

# Extract headers
header_row = rows[0]
clean_headers = [
    teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
    for cell in header_row.getElementsByType(TableCell)
]

# Extract data
clean_data = []
for row in rows[1:]:
    cells = row.getElementsByType(TableCell)
    values = [
        teletype.extractText(cell.getElementsByType(P)[0]).strip() if cell.getElementsByType(P) else ''
        for cell in cells
    ]
    # Pad values if row is short
    if len(values) < len(clean_headers):
        values += [''] * (len(clean_headers) - len(values))
    # Convert empty strings to None
    values = [v if v != '' else None for v in values]
    record = dict(zip(clean_headers, values))
    clean_data.append(record)

print(f"Loaded {len(clean_data)} records from cleaned_adult.ods")

Loaded 32561 records from cleaned_adult.ods


In [ ]:
from odf import text
from odf.opendocument import OpenDocumentSpreadsheet
from odf.table import Table, TableRow, TableCell
from odf.text import P
from collections import defaultdict

# calculating average
edu_hours = defaultdict(list)
for rec in clean_data:
  edu = rec.get('education')
  try:
    hours = float(rec.get('hours_per_week', 0) or 0)
    if edu and hours > 0:
      edu_hours[edu].append(hours)
  except Exception:
    continue
edu_avg = {edu: sum(hours)/len(hours) for edu, hours in edu_hours.items() if hours}

# Add education average to each record
enhanced_data = []
for rec in clean_data:
  edu = rec.get('education')
  avg = edu_avg.get(edu)
  enhanced_record = rec.copy()
  enhanced_record['education_avg_hours'] = round(avg, 2) if avg is not None else None
  enhanced_data.append(enhanced_record)

# create ods file with 2 sheets
doc = OpenDocumentSpreadsheet()
# sheet 1: cleaned data and average educationcolumn
table1 = Table(name= 'CleanedAdultWithEduAvgHours')
header_row1 = TableRow()
for h in clean_headers + ['education_avg_hours']:
  cell = TableCell()
  cell.addElement(P(text=h))
  header_row1.addElement(cell)
table1.addElement(header_row1)

for rec in enhanced_data:
  row = TableRow()
for h in clean_headers + ['education_avg_hours']:
  val = rec.get(h, '')
  cell = TableCell()
  cell.addElement(P(text= str(val) if val is not None else ''))
  row.addElement(cell)
  table1.addElement(row)
doc.spreadsheet.addElement(table1)

# sheet 2: Average hours by education
table2 = Table(name= 'AvgHoursByEducation')
header_row2 = TableRow()
for col in ['education', 'average_hours_per_week']:
  cell = TableCell()
  cell.addElement(P(text=col))
  header_row2.addElement(cell)
table2.addElement(header_row2)

for edu, avg in edu_avg.items():
  row = TableRow()
for val in [edu, round(avg, 2)]:
  cell = TableCell()
  cell.addElement(P(text=str(val)))
  row.addElement(cell)
table2.addElement(row)
doc.spreadsheet.addElement(table2)

task1_file = 'task1_avg_hours_by_education.ods'
doc.save(task1_file)
print(f'Task 1 file saved: {task1_file}')

Task 1 file saved: task1_avg_hours_by_education.ods


***Task 1: Verifier***

In [ ]:
from odf.opendocument import load
from odf.table import Table, TableRow, TableCell
from odf.text import P

def get_sheet_names(odf_doc):
    return[s.getAttribute('name') for s in odf_doc.getElementsByType(Table)]
def get_headers(sheet):
    header_row = sheet.getElementsByType(TableRow)[0]
    return[P.firstChild.data.strip() for P in header_row.getElementsByType(P) if P.firstChild]
def get_data_rows(sheet):
    return sheet.getElementsByType(TableRow)[1:]
def get_cell_text(cell):
    ps = cell.getElementsByType(P)
    return ps[0].firstChild.data.strip() if ps and ps[0].firstChild else ''
def validate_task1_full(task1_file, cleaned_file):
    print('===Task 1 validation===')
    cleaned_doc = load(cleaned_file)
    cleaned_sheet_names = get_sheet_names(cleaned_doc)
    print(f'Original Sheet Names: {cleaned_sheet_names}')
    cleaned_sheet = cleaned_doc.getElementsByType(Table)[0]
    cleaned_headers = get_headers(cleaned_sheet)
    print(f'Original Headers: {cleaned_headers}')
    doc = load(task1_file)
    task1_sheet_names = get_sheet_names(doc)
    print(f'- Task1 sheet names: {task1_sheet_names}')

    for name in cleaned_sheet_names:
      print(f'  Sheet {name} present in output: {name in task1_sheet_names}')
    table = [t for t in doc.getElementsByType(Table) if t.getAttribute('name') == 'AvgHoursByEducation']
    if not table:
      print("Sheet \'AvgHoursByEducation\' not found in output!")
      return
    table = table[0]
    headers = get_headers(table)
    print(f'Task1 headers: {headers}')
    header_names_match = headers = ['education', 'avarage_hours_per_week']
    print(f'- Header names match: {header_names_match}')
    print(f'- Header length match: {len(headers) == 2}')
    from collections import defaultdict
    edu_hours = defaultdict(list)
    for row in get_data_rows(cleaned_sheet):
      cells = row.getElementsByType(TableCell)
      row_dict = dict(zip(clean_headers, [get_cell_text(cell) for cell in cells]))
      try:
        edu = row_dict.get('education')
        hours = float(row_dict.get('hours_per_week', 0) or 0)
        if edu and hours > 0:
          edu_hours[edu].append(hours)
      except Exception:
        continue
    edu_avg = {edu: round(sum(hours)/len(hours), 2) for edu, hours in edu_hours.items() if hours}
    value_errors = 0
    for rows in get_data_rows(table):
      cells = rows.getElementsByType(TableCell)
      edu = get_cell_text(cells[0])
      avg = float(get_cell_text(cells[1]))
      expected = edu_avg.get(edu)
      if expected is not None and abs(avg - expected) > 0.01:
        print(f'Education: {edu} mismatch (expected {expected}, got {avg})')
        value_errors += 1
      print(f'- Average value validation errors: {value_errors}')
# Example usage:
validate_task1_full('task1_avg_hours_by_education.ods', 'cleaned_adult.ods')

===Task 1 validation===
Original Sheet Names: ['CleanedAdult']
Original Headers: ['age', 'workclass', 'fnlwgt', 'education', 'education_num', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income']
- Task1 sheet names: ['CleanedAdultWithEduAvgHours', 'AvgHoursByEducation']
  Sheet CleanedAdult present in output: False
Task1 headers: ['education', 'average_hours_per_week']
- Header names match: ['education', 'avarage_hours_per_week']
- Header length match: True
- Average value validation errors: 0
